# SUMO Network Heatmap Visualization
Interactive visualization of traffic congestion patterns from simulation results

## 1. Setup and Imports

In [1]:
import sys
from pathlib import Path

# Add parent directory to path
notebook_dir = Path.cwd()
if 'notebooks' not in str(notebook_dir):
    notebook_dir = Path.cwd().parent / 'notebooks'

sys.path.insert(0, str(notebook_dir))

# Import visualization module
from network_heatmap_visualization import (
    SUMONetworkParser, CongestionAnalyzer, HeatmapGenerator
)

print(f"Working directory: {Path.cwd()}")

ModuleNotFoundError: No module named 'network_heatmap_visualization'

## 2. Configure Paths

In [ ]:
from pathlib import Path

# Define base paths
repo_root = Path.cwd()
if 'notebooks' in str(repo_root):
    repo_root = repo_root.parent

# Available scenarios
scenarios = {
    'bgc_core': repo_root / 'scenarios' / 'bgc_core',
    'bgc_full': repo_root / 'scenarios' / 'bgc_full',
}

# Select scenario
selected_scenario = 'bgc_core'
scenario_path = scenarios[selected_scenario]

print(f"Selected scenario: {selected_scenario}")
print(f"Scenario path: {scenario_path}")
print(f"\nAvailable files:")
for f in sorted(scenario_path.glob('*.xml')):
    print(f"  - {f.name}")

## 3. Parse Network and Results

In [ ]:
# Path to network file
network_file = scenario_path / 'final_map.net.xml'

print(f"Parsing network: {network_file}")
network = SUMONetworkParser(str(network_file))

print(f"\nNetwork Statistics:")
print(f"  - Total edges: {len(network.edges)}")
print(f"  - Total junctions: {len(network.junctions)}")

# Get network bounds
edge_coords = network.get_edge_coordinates()
if edge_coords:
    xs = [c[0] for c in edge_coords.values()]
    ys = [c[1] for c in edge_coords.values()]
    print(f"\nNetwork Bounds:")
    print(f"  - X: [{min(xs):.2f}, {max(xs):.2f}]")
    print(f"  - Y: [{min(ys):.2f}, {max(ys):.2f}]")

## 4. Load Simulation Results

In [ ]:
# Available result files
result_files = list(scenario_path.glob('*.csv'))
result_files.extend(list(scenario_path.glob('data/*.csv')))

print("Available result files:")
for i, f in enumerate(result_files[:10], 1):
    print(f"  {i}. {f.name}")

# Select result file
results_file = scenario_path / 'baseline_results.csv'
if not results_file.exists():
    results_file = [f for f in result_files if 'baseline' in f.name][0]

print(f"\nSelected results file: {results_file.name}")

In [ ]:
import pandas as pd

# Parse results
print(f"Loading results from: {results_file}")
congestion = CongestionAnalyzer(str(results_file))

# Display summary
print(f"\nResults Summary:")
print(f"  - Time steps: {len(congestion.df)}")
print(f"  - Duration: {congestion.df['step'].max():.0f}s")
print(f"  - Max vehicles: {congestion.df['active_vehicles'].max():.0f}")
print(f"  - Max system wait: {congestion.df['total_system_wait'].max():.0f}")

# Show first few rows
print(f"\nFirst 10 time steps:")
display(congestion.df.head(10))

## 5. Analyze Congestion

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Get congestion statistics
peak_step, peak_congestion = congestion.get_peak_congestion()
avg_congestion = congestion.get_average_congestion()

print(f"Congestion Statistics:")
print(f"  - Average congestion: {avg_congestion:.2%}")
print(f"  - Peak congestion: {peak_congestion:.2%} at step {peak_step}")

# Categorize by congestion level
low = sum(congestion.df['combined_congestion'] < 0.3)
medium = sum((congestion.df['combined_congestion'] >= 0.3) & (congestion.df['combined_congestion'] < 0.7))
high = sum(congestion.df['combined_congestion'] >= 0.7)

print(f"\nCongestion Distribution:")
print(f"  - Low congestion (<30%): {low} steps")
print(f"  - Medium congestion (30-70%): {medium} steps")
print(f"  - High congestion (>70%): {high} steps")

## 6. Plot Congestion Over Time

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Configure style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

# Plot congestion metrics
fig, axes = plt.subplots(3, 1, figsize=(14, 10))

# Vehicles over time
axes[0].plot(congestion.df['step'], congestion.df['active_vehicles'], 
            linewidth=2, color='steelblue')
axes[0].fill_between(congestion.df['step'], congestion.df['active_vehicles'], 
                     alpha=0.3, color='steelblue')
axes[0].set_ylabel('Active Vehicles', fontsize=11, fontweight='bold')
axes[0].grid(True, alpha=0.3)
axes[0].set_title('Simulation Metrics Over Time', fontsize=13, fontweight='bold')

# System wait over time
axes[1].plot(congestion.df['step'], congestion.df['total_system_wait'], 
            linewidth=2, color='darkred')
axes[1].fill_between(congestion.df['step'], congestion.df['total_system_wait'], 
                     alpha=0.3, color='darkred')
axes[1].set_ylabel('Total System Wait (time units)', fontsize=11, fontweight='bold')
axes[1].grid(True, alpha=0.3)

# Congestion level over time
colors = plt.cm.RdYlGn_r(congestion.df['combined_congestion'])
axes[2].scatter(congestion.df['step'], congestion.df['combined_congestion'], 
               c=congestion.df['combined_congestion'], cmap='RdYlGn_r', 
               s=50, alpha=0.6, edgecolors='black', linewidth=0.5)
axes[2].plot(congestion.df['step'], congestion.df['combined_congestion'], 
            linewidth=1.5, color='gray', alpha=0.5)
axes[2].axhline(y=avg_congestion, color='blue', linestyle='--', linewidth=2, label=f'Average: {avg_congestion:.2%}')
axes[2].set_ylabel('Congestion Level', fontsize=11, fontweight='bold')
axes[2].set_xlabel('Simulation Step (s)', fontsize=11, fontweight='bold')
axes[2].set_ylim(0, 1)
axes[2].grid(True, alpha=0.3)
axes[2].legend(loc='upper left')

plt.tight_layout()
plt.savefig(Path.cwd().parent / 'heatmap_output' / 'simulation_metrics.png', dpi=150, bbox_inches='tight')
plt.show()

print("✓ Plot saved")

## 7. Generate Heatmaps

In [ ]:
# Create output directory
output_dir = Path.cwd().parent / 'heatmap_output'
output_dir.mkdir(exist_ok=True)

# Initialize heatmap generator
print(f"Generating heatmaps in: {output_dir}")
generator = HeatmapGenerator(network, congestion, str(output_dir))

In [ ]:
# Generate static heatmap
print("\n1. Generating static network heatmap...")
generator.generate_static_heatmap(figsize=(16, 12), dpi=150)

print("✓ Heatmap generated")

In [ ]:
# Display the static heatmap
from IPython.display import Image, display

heatmap_path = output_dir / 'network_heatmap_static.png'
if heatmap_path.exists():
    print("Network Heatmap:")
    display(Image(str(heatmap_path)))

In [ ]:
# Generate time series heatmap
print("\n2. Generating time series plot...")
generator.generate_time_series_heatmap(step_interval=20)

print("✓ Time series plot generated")

In [ ]:
# Display time series plot
ts_path = output_dir / 'congestion_time_series.png'
if ts_path.exists():
    print("Congestion Time Series:")
    display(Image(str(ts_path)))

In [ ]:
# Generate interactive map
print("\n3. Generating interactive map...")
map_path = generator.generate_interactive_folium_map()

print(f"✓ Interactive map saved to: {map_path}")
print(f"\nOpen in browser: {map_path}")

In [ ]:
# Generate summary report
print("\n4. Generating summary report...")
report_path = generator.generate_comparison_report()

# Read and display report
with open(report_path, 'r') as f:
    print(f.read())

## 8. Compare Multiple Scenarios

In [ ]:
# Compare different traffic levels
import os

data_dir = scenario_path / 'data'
csv_files = list(data_dir.glob('*.csv')) if data_dir.exists() else []

print(f"Found {len(csv_files)} CSV result files")
for f in csv_files:
    print(f"  - {f.name}")

In [ ]:
# Compare congestion across different result files
import pandas as pd
import matplotlib.pyplot as plt

comparison_data = {}

# Load multiple result files
result_files_to_compare = [
    scenario_path / 'baseline_results.csv',
]

# Add other result files if they exist
for f in [scenario_path / f for f in ['baseline_results_high.csv', 'baseline_results_low.csv', 'baseline_results_med.csv']]:
    if f.exists():
        result_files_to_compare.append(f)

print(f"Comparing {len(result_files_to_compare)} result files:\n")

fig, axes = plt.subplots(1, len(result_files_to_compare), figsize=(5*len(result_files_to_compare), 5))
if len(result_files_to_compare) == 1:
    axes = [axes]

for idx, result_file in enumerate(result_files_to_compare):
    if result_file.exists():
        analyzer = CongestionAnalyzer(str(result_file))
        label = result_file.stem.replace('baseline_results_', '').replace('baseline_results', 'baseline')
        
        print(f"{label}: avg congestion = {analyzer.get_average_congestion():.2%}")
        
        ax = axes[idx] if len(result_files_to_compare) > 1 else axes[0]
        ax.plot(analyzer.df['step'], analyzer.df['combined_congestion'], linewidth=2)
        ax.fill_between(analyzer.df['step'], analyzer.df['combined_congestion'], alpha=0.3)
        ax.set_title(label, fontweight='bold')
        ax.set_ylabel('Congestion Level')
        ax.set_xlabel('Time (s)')
        ax.set_ylim(0, 1)
        ax.grid(True, alpha=0.3)

plt.tight_layout()
comparison_path = output_dir / 'results_comparison.png'
plt.savefig(comparison_path, dpi=150, bbox_inches='tight')
plt.show()

print(f"\n✓ Comparison saved to: {comparison_path}")

## 9. Export Results Summary

In [ ]:
# Save detailed analysis to CSV
output_csv = output_dir / 'congestion_analysis.csv'
congestion.df.to_csv(output_csv, index=False)

print(f"✓ Analysis saved to: {output_csv}")

# Summary of outputs
print(f"\n=== VISUALIZATION OUTPUTS ===")
print(f"\nSaved files in {output_dir}:")
for f in sorted(output_dir.glob('*')):
    if f.is_file():
        size = f.stat().st_size / 1024  # KB
        print(f"  - {f.name} ({size:.1f} KB)")